# Task 1.2 — Key Assumptions
**Paper:** *Infinite SVM: A Dirichlet Process Mixture of Large-margin Kernel Machines* (ICML 2011)


## Assumption 1

**Assumption:** The true posterior over model parameters (z, η, γ, v) can be well-approximated by a fully factorised (mean-field) distribution q(z, η, γ, v) = ∏_d q(z_d) ∏_t q(η_t) q(γ_t) ∏_t q(v_t) with a finite truncation at level T.

**Why the method needs it:** The joint posterior p(z, η, γ, v | D) is intractable due to the infinite-dimensional DP and the coupling between classifier parameters and assignment variables. The mean-field assumption converts the inference problem into a tractable coordinate-descent optimisation over the variational parameters {ϕ, μ_t, ν_{t,1}, ν_{t,2}}.

**Violation scenario:** If the true posterior has strong correlations between component assignments z_d and classifier parameters η_t (e.g., when two components have nearly identical feature distributions but very different labels), the mean-field approximation may assign samples with high confidence to the wrong component, degrading both clustering and classification.

**Paper reference:** Section 3, opening paragraph: "we make the mean field assumption with truncated stick-breaking representations."

---

## Assumption 2

**Assumption:** The input features **x** are generated from an exponential family distribution (in practice, a Gaussian) with component-specific parameters γ_t, and the DP stick-breaking base distribution G₀ is the corresponding conjugate prior.

**Why the method needs it:** The coupling term in Eq. (11) — ρ · (E[γ_t]^⊤ **x**_d − E[A(γ_t)]) — is the expected log-likelihood of **x**_d under component t. If the feature distribution does not belong to an exponential family, this expectation has no closed form and the coordinate-descent update for ϕ becomes intractable.

**Violation scenario:** Real-world datasets with mixed data types (e.g., a combination of categorical, count, and continuous features) or heavy-tailed distributions (e.g., log-normal pixel intensities) violate the Gaussian assumption, causing the feature term to misrepresent cluster membership and producing poor soft assignments.

**Paper reference:** Section 2.3 (DP mixture of input features): "we consider the broad exponential family of distributions for x"; Section 4.2 specifies "we consider the real-valued features in iSVM by using normal distributions for x."

---

## Assumption 3

**Assumption:** The large-margin hinge-loss upper bound R(q(z, η)) (used in Eq. 6) is a valid and tight proxy for the true classification risk, even when classifier parameters η and component assignments z are learned jointly rather than independently.

**Why the method needs it:** iSVM avoids defining a normalised class-conditional likelihood p(y | **x**, z, η). Instead, it uses the MED hinge-loss (Eq. 6) as a surrogate risk. The entire learning framework—both the dual SVM QP (Eq. 10/12) and the φ-update (Eq. 11)—is derived under the assumption that minimising this hinge-loss upper bound leads to good generalisation.

**Violation scenario:** When class boundaries are highly ambiguous (very noisy labels, near-zero margin), the hinge loss becomes loose and the coupling between assignments and classifier margins breaks down. In extreme cases, the method may assign samples to components purely based on feature clusters that are irrelevant to the class boundary, rendering the large-margin principle ineffective.

**Paper reference:** Section 2.3: "R(q(z, η)) = Σ_d max_y (ℓ_d^Δ(y) + F(y, x_d) − F(y_d, x_d)) is an upper bound of the training error of rule (5)."

---

## Assumption 4

**Assumption:** The inductive test-time inference approximation—fixing q(v, γ) at the training posterior and running only one iteration of the q(z) variational update (Section 3.2)—is a sufficiently accurate approximation to the true test posterior p(z | **x**, D).

**Why the method needs it:** Exact test-time posterior inference requires iterating between q(z) and q(v, γ) until convergence, which would be computationally expensive. The one-iteration approximation makes prediction O(T) — fast and practical.

**Violation scenario:** When the test distribution is significantly different from the training distribution (dataset shift), fixing the training q(v, γ) means the component structure seen at training time may not match what is needed for the test point. A test sample from a new cluster not seen during training will be incorrectly forced into the nearest training component.

**Paper reference:** Section 3.2: "we approximate this procedure by using only one iteration, that is, we fix q(v, γ) at the distribution that approximates p(v, γ | D) inferred at training phase."
